## DOUAE

# BESOIN 3 : ARCHITECTURE DE CLASSIFICATION ET SAUVEGARDE DES IMAGES PNG

### 1. Importation des bibliothèques et configuration

Import des bibliothèques nécessaires :
- `pandas` / `numpy` : manipulation des données
- `joblib` : sérialisation du scaler et du modèle entraînés
- `matplotlib` / `seaborn` : visualisations statiques
- `scikit-learn` : `train_test_split`, `GridSearchCV`, `StandardScaler`, `LogisticRegression` et les métriques d'évaluation

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

## 2. Préparation et nettoyage des données

**Variable cible :** `implantation`.

**Variables retenues :** `puissance`, `nb_pdc`, `gratuit`, `prise_ccs`, `prise_type2`, `prise_chademo`, `prise_ef`, `prise_autre`, `deux_roues`.

**Justification du choix des variables :**
- `puissance` et `nb_pdc` : caractéristiques techniques de la borne, corrélées au type de lieu où elle est installée (ex. puissance élevée plutôt sur station dédiée).
- `gratuit`, `prise_ccs`, `prise_type2`, `prise_chademo`, `prise_ef`, `prise_autre`, `deux_roues` : indicateurs booléens du type d'équipement installé sur la borne (standards de prise disponibles, accessibilité deux-roues). Toutes ces variables sont des **choix techniques décidés au moment de la spécification de la borne**, indépendamment du contexte commercial du lieu — contrairement à des variables comme `condition_acces` ou `paiement_acte` qui sont elles des conséquences du lieu choisi (donc écartées, cf. discussion section 6.2 pour la justification de ce choix).

**Justification du nettoyage :** suppression des lignes avec valeurs manquantes (`dropna`) sur ces variables et la cible uniquement — une ligne incomplète sur l'une d'elles est inutilisable pour l'entraînement.

**Tableau récapitulatif des variables retenues :**

| Variable | Type | Rôle |
|---|---|---|
| `puissance` | numérique (kW) | caractéristique technique, corrélée au lieu d'installation |
| `nb_pdc` | numérique (entier) | taille de l'installation, corrélée au type de site |
| `gratuit` | booléen | modèle économique du lieu (voirie publique vs parking privé payant) |
| `prise_ccs` | booléen | standard de charge rapide, plus fréquent en station dédiée |
| `prise_type2` | booléen | standard généraliste, présent sur la majorité des sites |
| `prise_chademo` | booléen | standard rapide alternatif, indicateur de station dédiée |
| `prise_ef` | booléen | standard domestique (prise française), indicateur de borne simple/résidentielle |
| `prise_autre` | booléen | connecteur non standard, rare, indicateur de borne spécifique |
| `deux_roues` | booléen | accessibilité deux-roues, indicateur de borne urbaine/voirie |
| `implantation` (cible) | catégorielle (5 classes) | type de lieu à prédire |


In [2]:
df = pd.read_csv("export_IA.csv", low_memory=False)

colonne_cible = 'implantation'
colonnes_features = ['puissance', 'nb_pdc', 'gratuit', 'prise_ccs', 'prise_type2', 'prise_chademo',
                      'prise_ef', 'prise_autre', 'deux_roues']

df_propre = df.dropna(subset=colonnes_features + [colonne_cible]).copy()


### 2.1 Encodage des variables catégorielles

**Justification de l'encodage manuel des booléens :**
- Les colonnes `gratuit`, `prise_ccs`, `prise_type2`, `prise_chademo` arrivent sous des formats hétérogènes dans le CSV (`TRUE`/`FALSE`, `0`/`1`, parfois `OUI`/`NON`). Un mapping explicite vers `0`/`1` évite de dépendre du typage implicite de pandas, qui varie selon les valeurs réellement présentes.
- Les valeurs non reconnues sont mises à `0` (`fillna(0)`) plutôt que supprimées, car elles restent minoritaires et ne justifient pas de perdre la ligne entière.

In [3]:
colonnes_bool = ['gratuit', 'prise_ccs', 'prise_type2', 'prise_chademo', 'prise_ef', 'prise_autre', 'deux_roues']
for col in colonnes_bool:
    df_propre[col] = df_propre[col].astype(str).str.upper()
    df_propre[col] = df_propre[col].map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0, 'OUI': 1, 'NON': 0}).fillna(0).astype(int)


## 3. Analyse Exploratoire des Données (Visualisations)

**Justification du choix des graphiques :**
- **Boxplot puissance par implantation** : permet de visualiser si la distribution de `puissance` diffère selon le type de lieu — une variable qui ne discrimine pas les classes n'apporterait rien au modèle.
- **Barplot proportion de prise CCS par implantation** : la prise CCS est un standard de charge rapide, sa fréquence devrait varier fortement entre une station dédiée et un parking résidentiel — ce graphique valide cette hypothèse avant de l'utiliser comme feature.

In [4]:
print("Enregistrement des graphiques de justification...")

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_propre, x=colonne_cible, y='puissance', hue=colonne_cible, legend=False, palette='Set2')
plt.title("Justification de la variable 'puissance' selon l'Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("output/justification_puissance.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=df_propre, x=colonne_cible, y='prise_ccs', errorbar=None, hue=colonne_cible, legend=False, palette='Blues_r')
plt.title("Proportion de présence de Prises CCS par Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("output/proportion_prise_ccs.png", dpi=300)
plt.close()

Enregistrement des graphiques de justification...


# 4. Prétraitement des données (Preprocessing)

## 4.1 Séparation du jeu de données (Train/Test Split)

**Justification des paramètres :**
- `test_size=0.2` : répartition standard 80/20, suffisamment de données d'entraînement tout en gardant un jeu de test représentatif.
- `stratify=y` : conserve la proportion de chaque classe d'implantation dans les jeux d'entraînement et de test, important ici car les classes sont déséquilibrées (`Voirie` largement majoritaire face à `Parking privé réservé à la clientèle`).

In [5]:
X = df_propre[colonnes_features].astype(float)
y = df_propre[colonne_cible]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4.2 Normalisation des fonctionnalités et sauvegarde du Scaler

**Justification du choix de StandardScaler :**
- `puissance` et `nb_pdc` ont des échelles très différentes des variables booléennes (0/1) — sans normalisation, `LogisticRegression` accorderait un poids disproportionné aux variables à grande échelle.
- Le scaler est ajusté (`fit`) uniquement sur le jeu d'entraînement puis appliqué (`transform`) au jeu de test, pour éviter toute fuite d'information entre les deux.
- Le scaler est sauvegardé (`joblib`) car `main.py` doit appliquer exactement la même transformation aux nouvelles données, sans la réajuster.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler_pretraitement_b3.pkl')

['scaler_pretraitement_b3.pkl']

# 5. Entraînement du modèle et optimisation (GridSearchCV)

**Algorithmes comparés : Régression Logistique, Random Forest, Gradient Boosting, K-Nearest Neighbors**

**Justification de la comparaison :**
- La Régression Logistique seule plafonne autour de 0.51 d'accuracy : ses frontières de décision linéaires ne capturent pas les interactions entre `puissance`, `nb_pdc` et les types de prises.
- On entraîne et compare plusieurs familles de modèles avec `GridSearchCV`, puis on retient celui qui généralise le mieux.

**Méthodologie de sélection (correction d'un biais) :** le meilleur algorithme est choisi sur la base du score de validation croisée (`best_score_` de `GridSearchCV`, calculé uniquement sur les plis du jeu d'entraînement), **pas** sur l'accuracy du jeu de test. Utiliser le jeu de test pour choisir entre plusieurs modèles puis pour le reporter ensuite biaise l'estimation de performance (sur-optimisme, "winner's curse"). Le jeu de test n'est utilisé qu'une seule fois, en section 6, sur le modèle déjà sélectionné.

**Gestion du déséquilibre des classes :** la grille d'hyperparamètres de la Régression Logistique et du Random Forest inclut `class_weight: [None, 'balanced']` — `'balanced'` repondère automatiquement chaque classe par l'inverse de sa fréquence, pour que le modèle ne soit pas mécaniquement biaisé vers `Voirie` (classe majoritaire) au détriment de `Parking privé réservé à la clientèle` (classe rare, 307 individus). `GradientBoostingClassifier` et `KNeighborsClassifier` ne supportent pas nativement `class_weight` dans scikit-learn — limite connue de ces implémentations, conservée ici par souci de simplicité.

**Principe de fonctionnement — Régression Logistique :**
Modélise pour chaque classe une combinaison linéaire des variables (`z = w·x + b`), transformée par une fonction softmax en probabilité d'appartenance. Frontières de décision linéaires (hyperplans).

**Principe de fonctionnement — Random Forest :**
Construit un grand nombre d'arbres de décision sur des échantillons bootstrap et des sous-ensembles aléatoires de variables. Prédiction finale = vote majoritaire. Capture nativement les interactions et seuils non linéaires.

**Principe de fonctionnement — Gradient Boosting :**
Construit les arbres séquentiellement, chacun corrigeant les erreurs des précédents, pondérées par `learning_rate`.

**Principe de fonctionnement — K-Nearest Neighbors :**
Prédit la classe majoritaire parmi les `k` voisins les plus proches (distance euclidienne sur variables normalisées).

In [7]:
resultats_algos = {}

print("Recherche des meilleurs hyperparamètres (Régression Logistique)...")
param_grid_lr = {'C': [0.1, 1.0, 10.0], 'solver': ['lbfgs', 'saga'], 'max_iter': [200], 'class_weight': [None, 'balanced']}
grid_lr = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)
resultats_algos['Régression Logistique'] = {'modele': grid_lr.best_estimator_, 'params': grid_lr.best_params_, 'cv_score': grid_lr.best_score_}

print("Recherche des meilleurs hyperparamètres (Random Forest)...")
param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [10, 20, None], 'min_samples_leaf': [1, 5], 'class_weight': [None, 'balanced']}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)
resultats_algos['Random Forest'] = {'modele': grid_rf.best_estimator_, 'params': grid_rf.best_params_, 'cv_score': grid_rf.best_score_}

print("Recherche des meilleurs hyperparamètres (Gradient Boosting)...")
param_grid_gb = {'n_estimators': [100], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5]}
grid_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid_gb, cv=5, scoring='accuracy', n_jobs=-1)
grid_gb.fit(X_train_scaled, y_train)
resultats_algos['Gradient Boosting'] = {'modele': grid_gb.best_estimator_, 'params': grid_gb.best_params_, 'cv_score': grid_gb.best_score_}

print("Recherche des meilleurs hyperparamètres (K-Nearest Neighbors)...")
param_grid_knn = {'n_neighbors': [5, 9, 15], 'weights': ['uniform', 'distance']}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train)
resultats_algos['K-Nearest Neighbors'] = {'modele': grid_knn.best_estimator_, 'params': grid_knn.best_params_, 'cv_score': grid_knn.best_score_}

print("\nCOMPARAISON DES MODÈLES (accuracy en validation croisée, jeu d'entraînement uniquement) :")
print('=' * 75)
for nom, info in resultats_algos.items():
    print(f"{nom:25s} -> cv_score = {info['cv_score']:.4f}  | params = {info['params']}")

meilleur_nom = max(resultats_algos, key=lambda n: resultats_algos[n]['cv_score'])
meilleur_modele = resultats_algos[meilleur_nom]['modele']
print(f"\nModèle retenu (sur score CV, jeu de test pas encore consulté) : {meilleur_nom} "
      f"(cv_score = {resultats_algos[meilleur_nom]['cv_score']:.4f})")


Recherche des meilleurs hyperparamètres (Régression Logistique)...


/home/isen/.local/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/home/isen/.local/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/home/isen/.local/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/isen/.local/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/home/isen/.local/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


...

/home/isen/.local/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Recherche des meilleurs hyperparamètres (Random Forest)...


Recherche des meilleurs hyperparamètres (Gradient Boosting)...


Recherche des meilleurs hyperparamètres (K-Nearest Neighbors)...



COMPARAISON DES MODÈLES (accuracy en validation croisée, jeu d'entraînement uniquement) :
Régression Logistique     -> cv_score = 0.5119  | params = {'C': 0.1, 'class_weight': None, 'max_iter': 200, 'solver': 'lbfgs'}
Random Forest             -> cv_score = 0.7527  | params = {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
Gradient Boosting         -> cv_score = 0.7480  | params = {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100}
K-Nearest Neighbors       -> cv_score = 0.7387  | params = {'n_neighbors': 15, 'weights': 'distance'}

Modèle retenu (sur score CV, jeu de test pas encore consulté) : Random Forest (cv_score = 0.7527)


# 6. Évaluation du modèle final

**Justification du choix des métriques :**
- `classification_report` donne précision, rappel et F1-score par classe — indispensable ici car les classes sont déséquilibrées : l'accuracy globale seule masquerait une mauvaise prédiction sur les classes minoritaires.

In [8]:
y_pred = meilleur_modele.predict(X_test_scaled)

print("RAPPORT D'ÉVALUATION DE LA CLASSIFICATION")
print(classification_report(y_test, y_pred))

RAPPORT D'ÉVALUATION DE LA CLASSIFICATION
                                      precision    recall  f1-score   support

Parking privé réservé à la clientèle       0.90      0.52      0.66       307
        Parking privé à usage public       0.76      0.78      0.77      7499
                      Parking public       0.81      0.57      0.67      4110
 Station dédiée à la recharge rapide       0.73      0.70      0.72      1477
                              Voirie       0.74      0.84      0.78      9185

                            accuracy                           0.76     22578
                           macro avg       0.79      0.68      0.72     22578
                        weighted avg       0.76      0.76      0.75     22578



In [9]:
rapport_dict = classification_report(y_test, y_pred, output_dict=True)
rapport_df = pd.DataFrame(rapport_dict).transpose().round(2)
rapport_df


,precision,recall,f1-score,support
Parking privé réservé à la clientèle,0.90,0.52,0.66,307.00
Parking privé à usage public,0.76,0.78,0.77,7499.00
Parking public,0.81,0.57,0.67,4110.00
Station dédiée à la recharge rapide,0.73,0.70,0.72,1477.00
Voirie,0.74,0.84,0.78,9185.00
accuracy,0.76,0.76,0.76,0.76
macro avg,0.79,0.68,0.72,22578.00
weighted avg,0.76,0.76,0.75,22578.00


## 6.1 Matrice de confusion

**Justification :** complète le rapport de classification en visualisant précisément quelles classes sont confondues entre elles — utile ici pour identifier que `Parking public` est largement confondu avec `Voirie` (cf. discussion plus bas).

In [10]:
matrice = confusion_matrix(y_test, y_pred)
classes = meilleur_modele.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(matrice, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Matrice de Confusion - Implantation de la Station", fontsize=12, fontweight='bold')
plt.ylabel('Réalité Terrain')
plt.xlabel('Prédiction Modèle')
plt.tight_layout()
plt.savefig("output/matrice_confusion.png", dpi=300)
plt.close()

## 6.1bis Importance des variables (Feature Importance)

**Justification :** vérifie empiriquement que les 9 variables retenues apportent réellement du signal au modèle final, plutôt que de se fier uniquement à l'intuition métier de la section 2. Disponible nativement pour les modèles à base d'arbres (`feature_importances_`) ; pour un modèle linéaire, on affiche les coefficients à la place (valeur absolue = poids dans la décision).

In [11]:
if hasattr(meilleur_modele, 'feature_importances_'):
    importances = pd.Series(meilleur_modele.feature_importances_, index=colonnes_features).sort_values(ascending=False)
    titre = f"Importance des variables - {meilleur_nom}"
elif hasattr(meilleur_modele, 'coef_'):
    importances = pd.Series(np.abs(meilleur_modele.coef_).mean(axis=0), index=colonnes_features).sort_values(ascending=False)
    titre = f"Importance des variables (|coefficients| moyens) - {meilleur_nom}"
else:
    importances = None

if importances is not None:
    print(importances)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False, palette='viridis')
    plt.title(titre, fontsize=12, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig("output/feature_importance.png", dpi=300)
    plt.close()
else:
    print(f"{meilleur_nom} n'expose ni feature_importances_ ni coef_ (ex. KNN) - pas d'importance disponible.")


puissance        0.443117
nb_pdc           0.372913
prise_type2      0.063553
prise_ef         0.051469
prise_ccs        0.035273
prise_chademo    0.013425
prise_autre      0.008534
deux_roues       0.007152
gratuit          0.004565
dtype: float64


## 6.2 Discussion des résultats

**Itération supplémentaire :** 3 variables techniques ajoutées (`prise_ef`, `prise_autre`, `deux_roues`), en plus des 6 initiales — gain d'accuracy en Random Forest (0.7263 → ~0.75), mesuré empiriquement avant intégration.

**Pourquoi seulement ces 3 et pas plus :** d'autres variables (`condition_acces`, `paiement_acte/cb/autre`, `type_tarif`, `restriction_gabarit`) auraient pu pousser l'accuracy jusqu'à ~0.88, mais elles ont été **volontairement écartées** : ce sont des conséquences du choix d'implantation, pas des causes indépendantes — les inclure aurait été un raisonnement circulaire, sans utilité pour `main.py` qui doit aider à choisir l'implantation d'une borne **avant** son installation.

**Comparaison des 4 algorithmes — score de validation croisée (jeu d'entraînement uniquement) :**

| Algorithme | CV score (train) |
|---|---|
| Régression Logistique | 0.5119 |
| Random Forest | **0.7527** |
| Gradient Boosting | 0.7480 |
| K-Nearest Neighbors | 0.7387 |

**Modèle retenu : Random Forest**, choisi sur ce score de validation croisée — le jeu de test n'a servi qu'une seule fois ensuite, pour l'évaluation finale (accuracy test = 0.76, cohérent avec le score CV de 0.7527, ce qui confirme l'absence de sur-optimisme dans la sélection).

**`class_weight='balanced'` testé mais pas retenu :** `GridSearchCV` a comparé `None` vs `'balanced'` pour la Régression Logistique et le Random Forest ; le meilleur score CV a été obtenu avec `class_weight=None` pour les deux. Le rééquilibrage n'a donc pas amélioré la performance globale ici — résultat empirique, pas un choix a priori.

**Importance des variables (Random Forest) :** `puissance` (44%) et `nb_pdc` (37%) dominent largement, suivies de loin par `prise_type2` (6%) et `prise_ef` (5%). `prise_chademo`, `prise_autre`, `deux_roues` et `gratuit` contribuent chacune moins de 1.5% — elles apportent un signal marginal mais réel (retirées, l'accuracy redescendrait légèrement vers le niveau de la version à 6 variables).

**Par classe (Random Forest, test) :**
- `Voirie` (classe majoritaire, ~9185 individus) : recall 0.84, f1 0.78.
- `Parking public` : recall 0.57 — point faible persistant.
- `Parking privé réservé à la clientèle` : précision 0.90, recall 0.52 — classe rare (307/22578), sous-détectée malgré le test de `class_weight='balanced'`.
- `Station dédiée à la recharge rapide` : f1 0.72.

**Limite principale :** la classe rare reste sous-détectée même après avoir testé le rééquilibrage des classes. Pistes restantes : SMOTE (sur-échantillonnage synthétique, différent de `class_weight`) ou collecte de données supplémentaires sur cette classe.

# 7. Exportation du modèle prédictif final

**Justification :** le modèle est sauvegardé (`joblib`) pour permettre à `main.py` de charger directement le modèle entraîné sans jamais relancer d'apprentissage à l'exécution.

In [12]:
joblib.dump(meilleur_modele, 'modele_classification_b3.pkl')


['modele_classification_b3.pkl']